In [1]:
import pandas as pd
from pathlib import Path

base = Path(".") if Path("prices.csv").exists() else Path("data")
prices = pd.read_csv(base / "prices.csv", parse_dates=["date"])
index = pd.read_csv(base / "market_index.csv", parse_dates=["date"])
port = pd.read_csv(base / "portfolio.csv")

print("PRICES:", prices.shape)
print("Date range:", prices["date"].min().date(), "to", prices["date"].max().date())
print("Tickers:", prices["ticker"].nunique())

# 1. Portfolio tickers missing from prices
missing = set(port["ticker"]) - set(prices["ticker"])
print("\n1. Tickers in portfolio but NOT in prices:", missing or "none (good)")

# 2. Per-ticker summary
summary = prices.groupby("ticker").agg(
    rows=("date", "count"),
    first=("date", "min"),
    last=("date", "max"),
    zero_volume_days=("volume", lambda v: (v == 0).sum()),
    missing_values=("close", lambda v: v.isna().sum()),
)
print("\n2. Per-ticker summary:")
display(summary)

# 3. Bad rows
bad_hl = (prices["high"] < prices["low"]).sum()
print("\n3. Rows where High < Low:", bad_hl)
print("   Rows with missing High/Low/Volume:",
      prices[["high", "low", "volume"]].isna().any(axis=1).sum())

# 4. Market index
print("\n4. MARKET INDEX:", index.shape)
print(index.isna().sum())

# 5. Quick liquidity preview: shares vs 20-day average volume
adv = (prices.sort_values("date").groupby("ticker")["volume"]
       .apply(lambda v: v.tail(20).mean()).rename("adv20"))
check = port.merge(adv, left_on="ticker", right_index=True, how="left")
check["days_at_10pct"] = (check["shares"] / (0.10 * check["adv20"])).round(1)
print("\n5. Days to liquidate at 10% participation:")
display(check.sort_values("days_at_10pct", ascending=False))

PRICES: (42761, 8)
Date range: 2018-01-01 to 2026-10-23
Tickers: 20

1. Tickers in portfolio but NOT in prices: none (good)

2. Per-ticker summary:


,rows,first,last,zero_volume_days,missing_values
ticker,,,,,
HDFCBANK.NS,2176,2018-01-01,2026-10-23,2,0
HFCL.NS,2176,2018-01-01,2026-10-23,2,0
ICICIBANK.NS,2176,2018-01-01,2026-10-23,2,0
IDEA.NS,2176,2018-01-01,2026-10-23,2,0
INFY.NS,2176,2018-01-01,2026-10-23,2,0
IRFC.NS,1417,2021-01-29,2026-10-23,3,0
ITC.NS,2176,2018-01-01,2026-10-23,1,0
JPPOWER.NS,2176,2018-01-01,2026-10-23,2,0
LT.NS,2176,2018-01-01,2026-10-23,2,0



3. Rows where High < Low: 0
   Rows with missing High/Low/Volume: 0

4. MARKET INDEX: (2173, 3)
date          0
nifty         2
india_vix    17
dtype: int64

5. Days to liquidate at 10% participation:


,ticker,shares,sector,adv20,days_at_10pct
19,TRIDENT.NS,25000000,Textiles,3.128804e+06,79.9
9,VOLTAS.NS,2500000,Consumer Durables,6.969399e+05,35.9
10,NHPC.NS,30000000,Power,8.834626e+06,34.0
11,IRFC.NS,30000000,Railways,1.131117e+07,26.5
17,HFCL.NS,20000000,Telecom Equipment,8.355622e+06,23.9
12,PNB.NS,40000000,Banking,1.962031e+07,20.4
16,RPOWER.NS,50000000,Power,2.513759e+07,19.9
18,JPPOWER.NS,80000000,Power,4.697976e+07,17.0
14,SUZLON.NS,120000000,Power Equipment,7.138763e+07,16.8
15,YESBANK.NS,200000000,Banking,1.405221e+08,14.2


In [2]:
import plotly.express as px
import plotly.graph_objects as go

### Data Completeness (Do all data have full history ?)

In [3]:
counts = prices.groupby("ticker").size().reset_index(name="rows").sort_values("rows")
fig = px.bar(counts, x="rows", y="ticker", orientation="h",
             title="Rows of data per stock (IRFC is shorter: listed Jan 2021)")
fig.show()

### price history, all stocks rebased to 100

In [4]:
p = prices.dropna(subset=["close"]).sort_values(["ticker", "date"]).copy()
p["rebased"] = p.groupby("ticker")["close"].transform(lambda s: s / s.iloc[0] * 100)
fig = px.line(p, x="date", y="rebased", color="ticker", log_y=True,
              title="Price history (start = 100, log scale)")
fig.show()

### how much each stock trades per day (liquidity)

In [5]:
p2 = prices[prices["volume"] > 0].copy()
p2["traded_value_cr"] = p2["close"] * p2["volume"] / 1e7
tv = (p2.groupby("ticker")["traded_value_cr"].median()
        .reset_index().sort_values("traded_value_cr"))
fig = px.bar(tv, x="traded_value_cr", y="ticker", orientation="h", log_x=True,
             title="Median daily traded value (₹ crore, log scale)")
fig.show()

### days to liquidate at 10% participation

In [6]:
last = (prices.sort_values("date").groupby("ticker").tail(20)
        .groupby("ticker")["volume"].mean().rename("adv20"))
chk = port.merge(last, left_on="ticker", right_index=True)
chk["days_at_10pct"] = chk["shares"] / (0.10 * chk["adv20"])
chk["bucket"] = pd.cut(chk["days_at_10pct"], [0, 3, 12, 40, 1e9],
                       labels=["Easy (<3d)", "Medium (3-12d)", "Hard (12-40d)", "Very hard (>40d)"])
chk = chk.sort_values("days_at_10pct")
fig = px.bar(chk, x="days_at_10pct", y="ticker", color="bucket", orientation="h",
             title="Days to liquidate each position (10% of daily volume)")
fig.show()

### Nifty and India VIX with the stress line

In [7]:
idx = index.dropna(subset=["nifty", "india_vix"])
thr = idx["india_vix"].quantile(0.80)
fig = go.Figure()
fig.add_trace(go.Scatter(x=idx["date"], y=idx["nifty"], name="Nifty 50"))
fig.add_trace(go.Scatter(x=idx["date"], y=idx["india_vix"], name="India VIX", yaxis="y2"))
fig.add_hline(y=thr, line_dash="dash", yref="y2",
              annotation_text=f"Stress line: VIX > {thr:.1f}")
fig.update_layout(title="Market index and fear gauge (Covid peak in March 2020)",
                  yaxis=dict(title="Nifty"),
                  yaxis2=dict(title="India VIX", overlaying="y", side="right"))
fig.show()